In [ ]:
# ── Run this chapter on a clean machine (Colab "Julia" runtime, Binder, or any Jupyter with a Julia kernel) ──
# Cell 1 of 2 — the engine and the data. Measured on a clean machine: about three minutes to a first fit.
# Both engines are public on GitHub, so nothing needs a registry.
import Pkg
Pkg.add(url = "https://github.com/itchyshin/DRM.jl", rev = "26f4c4ddca58bd672fce807a7564face3debdc93")   # the commit the chapters were executed against
Pkg.add(["DataFrames", "CSV", "Distributions", "StatsBase", "StatsModels"])
REPO_RAW = "https://raw.githubusercontent.com/itchyshin/stats-hours/main"   # works once the repository is public
for f in ("tools/theme_itchy.jl", "tools/figures.jl", "tools/engine-pin.txt", "data/2012/MBodySize.csv", "data/2012/BodySize.csv", "data/2012/ChickSurvival.csv", "data/2012/FemaleSuccess.csv", "data/2012/SparrowSurvival.csv")
    mkpath(dirname(f)); isfile(f) || download("$REPO_RAW/$f", f)
end
println("engine and data ready — run the next cell for the plotting stack (several minutes; read on meanwhile)")

In [ ]:
# Cell 2 of 2 — the plotting stack. This is the slow part on a bare machine (about eight minutes measured;
# Binder pays it once at image build, so there it is seconds). Every figure in the chapter needs it.
import Pkg
Pkg.add(["Makie", "CairoMakie", "AlgebraOfGraphics"])
using CairoMakie
println("plotting ready")

---
title: "Class 1: base camp"
book: Stats Hours with Itchy
chapter: 1
type: book-chapter
status: draft
created: 2026-09-07
engines: DRM.jl 0.7.1 at 26f4c4ddc (Julia 1.10.0) · drmTMB 0.7.0 (R 4.6.0)
tags: [book, julia, dataframes, plots, errors, seeds, DRM.jl]
deck: "Before any model there is a file, a data frame, a picture, and a wall of red text. The wall of red is the part that makes people give up, so it is the part we teach first."
status_tag: Draft
status_note: "Draft. Every number and figure on this page comes from code that was actually run when the site was built; the prose is still being written."
provenance: "Everything printed below — tables, figures and the four errors — came from running this chapter's code from top to bottom when the site was built; nothing is typed from memory. The four errors are real errors, left in on purpose. Every random draw states its seed right where it is made."
caveat: "The data are real: 171 house sparrows measured on Lundy Island in the author's 2012 course, read directly from data/2012/MBodySize.csv (where the file came from is recorded in data/2012/README.md). Nothing here is simulated except the coin flips, which say so and carry their seeds."
footer_note: "Stats Hours with Itchy · Class 1 of twelve rungs (ten in version 1, plus a coda) · draft · code last run 2026-09-07"
---

# Class 1: base camp

> **What this chapter does not teach: installation.** You need Julia running and this book's
> packages available, and there are two honest routes. The **hosted notebook** needs nothing at
> all — open it and the first cell below works. A **local install** needs the two engines pulled
> from their repositories by URL, which is a paragraph of instructions that will change the day
> the two packages are listed in Julia's public package catalogue and become a one-line `Pkg.add`. Writing that paragraph now
> means writing it twice, so this chapter does not. Everything from the first cell onward assumes
> you are already at a prompt.

---

## Objectives

By the end of this class you should be able to:

1. Read a CSV file (comma-separated values — a plain-text spreadsheet) into a data frame (a table with named columns, one row per bird), and say what `using` does that R's `library` does not.
2. Look at a data frame four ways — its size, its first rows, its column types, its summary — and say what each of the four would catch that the others would miss.
3. Ask the rows a question: filter them, group them, and compute a mean per group.
4. Draw the data on its own, before any model, in the book's house figure grammar.
5. Make a random draw that is the same tomorrow as it is today, and say precisely what the seed does and does not promise.
6. **Read a Julia error message**: find the line that says what you asked for, the line that says what exists, and the marker that points at the argument you got wrong.

---

## The class

**Itchy's office, 9:00 am on the first morning. Four chairs, one whiteboard, and a laptop belonging to TOTO on which Julia has been installed for eleven minutes. MOMO has arrived from a little R, used under protest, and is already suspicious of anything that claims to be better. EDDIE, who has done this before, is here for the coffee.**

**Itchy:** Nobody fits anything today. Today you get a file into Julia, look at it, draw it, and break it on purpose four times. That last part is not a joke and it is not filler — it is the reason people give up on this language in week one, and I would rather you met it here, with me, than alone at eleven at night.

**Momo:** In R I opened a file with `read.csv` and never thought about it again.

**Itchy:** And in an hour you will open one here with a line that tells you what every column is before you touch it, and you will have watched it fail four times and known each time why. That is the trade: a little more looking now, a great deal less guessing later. Toto, type this.

In [ ]:
#| label: setup
# the book's plotting helpers and colours
include("tools/figures.jl")
using DRM, DataFrames, CSV, Statistics, Random, Printf, CairoMakie
set_theme!(theme_itchy(:light))

sparrows = CSV.read("data/2012/MBodySize.csv", DataFrame)

n_birds = nrow(sparrows)
n_cols = ncol(sparrows)

@printf("rows: %d   columns: %d\n", n_birds, n_cols)
println(names(sparrows))

**Toto:** It printed the column names.

**Itchy:** `{julia} n_birds` sparrows, `{julia} n_cols` columns, measured on Lundy Island, and every one of them was a real bird somebody held. Now the boring line, because it is the one that goes wrong most.

**Momo:** The `using` line?

**Itchy:** The `using` line. In R, `library(dplyr)` says "make dplyr's functions visible". `using DataFrames` says the same thing and one thing more: it is also the moment Julia **compiles** what it needs. The first `using` in a session is slow — seconds, sometimes tens of seconds — and every one after it in that session is instant. If you type it and the room goes quiet, nothing is broken. That is the whole of the famous complaint about this language, and it happens once.

**Eddie:** It is the only time all hour the machine looks stupider than R.

**Itchy:** It is, and I would rather say so than have you discover it. The other line worth reading slowly is `CSV.read`. Two packages do that job: `CSV` knows how to parse a file, `DataFrames` knows what a table is, and you hand the first the second as an argument. In R those live in one function. Here they are two ideas and you can see the join.

**Toto:** And the path has two dots at the front.

**Itchy:** Because this chapter lives in `book/` and the data live in `data/`, so the path climbs out one level. Paths are relative to the file being run, and that is what makes this page reproducible on a machine that is not mine.

**Toto:** And the line that printed the counts starts with an `@`.

**Itchy:** The `@` marks a **macro**: a thing that rewrites the code you hand it before running it, which lets it do what a function cannot. You will meet one more, `@formula`, before we finish. Read `@printf` as "print, to a pattern": `%d` is a slot for a whole number, `%.4f` a decimal with four places, `\n` ends the line, and the values after the comma fill the slots in order. `println` prints what it is given, with no pattern.

### Look at it four ways

**Itchy:** Before anything else, look. Four different looks, and each one catches something the other three do not.

In [ ]:
#| label: look-first
first(sparrows, 5)

**Toto:** There is a row of words under the column names.

**Itchy:** That row is the best thing on this page and R does not print it. `Tarsus` is a `Float64` — a number with a decimal point. `Sex` is a `String1` — text, one character wide. `BirdID` is a `String7`. Julia is telling you the **type** of every column before you have asked, and type is what determines what you are allowed to do next. Half of the errors in this class are a type you did not check.

**Momo:** In R a column had a type too, and it lied to me constantly.

**Itchy:** Here it cannot lie, because it is not a guess made silently when the file was read; it is what is actually in memory, printed. If a column you expected to be numeric prints as `String`, you have found a stray comma or the word `NA` in row four hundred, and you have found it in the first five seconds rather than in the middle of a model. Now the summary.

In [ ]:
#| label: look-describe
describe(sparrows, :mean, :std, :min, :max, :nmissing)

**Momo:** It did not refuse on the text columns.

**Itchy:** It did not refuse; it left the arithmetic blank and still told you the smallest and largest values, which for `Sex` are the two letters and for `BirdID` are the first and last identifiers alphabetically. That is `describe` being useful rather than fussy. The column I actually care about is `nmissing`, and every entry in it is zero, so this file has no holes in it. And notice how you asked for them: `:mean`, `:nmissing`, a word with a colon in front. That is a **Symbol**, a name written down as a value, so a function can be handed the *name* of a thing without Julia looking the thing up. DataFrames uses one wherever a column is referred to rather than read. Say the numbers out loud, Toto.

In [ ]:
#| label: look-numbers
mean_wing = mean(sparrows.Wing)
sd_wing = std(sparrows.Wing)
tarsus_min, tarsus_max = extrema(sparrows.Tarsus)

@printf("mean wing   : %.4f mm\n", mean_wing)
@printf("SD of wing  : %.4f mm\n", sd_wing)
@printf("tarsus range: %.4f to %.4f mm\n", tarsus_min, tarsus_max)

**Toto:** The average wing is `{julia} round(mean_wing, digits = 2)` millimetres.

**Itchy:** With a standard deviation — the typical distance of one bird from that average — of `{julia} round(sd_wing, digits = 2)`, and tarsus — the leg bone, the standard measure of how big a sparrow is — running from `{julia} round(tarsus_min, digits = 2)` to `{julia} round(tarsus_max, digits = 2)`. Note the dot in `sparrows.Wing`. That is how you reach a column, and what comes back is a plain vector of numbers you can hand to any function in the language. Not a special data-frame object with its own rules. A vector. And `extrema` returned two numbers at once, which is why that line has two names on the left with a comma between them: a function may return several things, and you catch them all.

**Eddie:** Which is why `mean` works on it without knowing what a data frame is.

**Itchy:** Exactly why, and it is worth a moment. `mean` comes from `Statistics`, `sparrows.Wing` comes from `DataFrames`, and neither package has heard of the other. They agree on `Vector{Float64}` and that is enough. You will meet that pattern all the way to Class 10. Now the fourth look, which is a picture, because six summary numbers can be identical for data of wildly different shapes.

In [ ]:
#| label: fig-wing-hist
#| fig-cap: "The distribution of wing length, which the six summary numbers cannot show you."
fig = Figure(size = (540, 360))
ax = Axis(fig[1, 1]; xlabel = "wing (mm)", ylabel = "sparrows",
    title = "one column, drawn")
# colour chosen by hand so the histogram matches the book's other figures
hist!(ax, sparrows.Wing; bins = 20, color = LIGHT.accent)
vlines!(ax, [mean_wing]; linestyle = :dash, color = (:grey, 0.8))
fig

**Itchy:** One hump, roughly symmetric, with a small shoulder further out in the right tail that keeps it from tapering away evenly, nothing stranded at zero because somebody typed a missing value as a number. The dashed line is the mean. Look at this before you believe any summary; it takes four seconds and it is the cheapest thing in this book.

Two things about how that cell is written, because every figure in this book is written the same way. Everything after the semicolon in a call — `xlabel`, `bins`, `color` — is a **keyword argument**: named, any order, and the semicolon marks where the positional ones stop. And the `!` on `hist!` is a convention, not syntax: it warns that the function changes something you handed it, here by drawing onto `ax`.

### The first figure of the grammar

**Itchy:** Now two columns at once. This book has a small set of house figures, and the first of them is the one you draw before you have a model: the **data cloud**.

In [ ]:
#| label: fig-cloud
#| fig-cap: "The data cloud: wing length against tarsus, with nothing fitted to it."
fig_data_cloud(sparrows.Tarsus, sparrows.Wing;
    xlabel = "tarsus (mm)", ylabel = "wing (mm)",
    title = "171 Lundy sparrows, and no model")

**Toto:** There is obviously a line through it.

**Itchy:** There is obviously *something* through it, and the entire point of drawing this before Class 2 is that you saw the pattern with your own eyes and not through a *p*-value — the number software gives you for how surprising a pattern this strong would be if there were really nothing there. It arrives in Class 2; today you do not need it. Next week we put a line there and it will be the right thing to do. But the habit — cloud first, model second — is the habit that stops you fitting a straight line to something that is plainly a curve, which is a mistake I have watched people publish.

**Momo:** And `fig_data_cloud` is not a Julia function.

**Itchy:** It is a Julia function, it is just ours, and it lives in `tools/figures.jl` — the file the first cell included. It wraps the plotting library and the book's colours so that every figure in every chapter looks like it came from the same book. When you write your own, use whatever you like; when you read this book, that is what the house style is made of.

### Ask the rows a question

**Itchy:** Data frames earn their keep when you ask them something. Three verbs, and then a comparison to R that I will not repeat all year.

In [ ]:
#| label: filter-and-subset
males = filter(row -> row.Sex == "M", sparrows)
females = subset(sparrows, :Sex => ByRow(==("F")))

n_males = nrow(males)
n_females = nrow(females)

@printf("males  : %d\n", n_males)
@printf("females: %d\n", n_females)
@printf("total  : %d  (should be %d)\n", n_males + n_females, n_birds)

**Toto:** Two different ways to do the same thing.

**Itchy:** Two ways with different characters. `filter` is base Julia's own verb and it takes a **function first** and the thing to be filtered second — `filter(row -> row.Sex == "M", sparrows)`. That little arrow makes an anonymous function: "given a row, is its `Sex` equal to `M`". If you have used R's `dplyr`, this is the one place the two languages do not merely differ but *reverse*: `filter(data, condition)` there, function first here. Every R user types it the wrong way round once, and you will do it later in the hour, on purpose. `subset` is the DataFrames verb, takes the frame first, and reads closer to `dplyr` once you know two things: `==("F")` is a comparison with one side left open, which makes it a function asking "is this equal to F", and `ByRow` says to ask that of every row in turn. Use either. Do not mix them up in the same script and expect to be able to read it in a year.

**Momo:** And you get `{julia} n_males` and `{julia} n_females`, which add up.

**Itchy:** Which add up to `{julia} n_birds`, and I want you to notice that I checked. Every time you split something, add the pieces back up. It costs one line and it has caught more of my mistakes than any diagnostic in Class 5a. Now the grouped question, which is the one you will actually use.

In [ ]:
#| label: grouped-mean
by_sex = combine(groupby(sparrows, :Sex),
                 :Wing => mean => :mean_wing,
                 :Wing => std => :sd_wing,
                 nrow => :n)
wing_f = by_sex.mean_wing[by_sex.Sex .== "F"][1]
wing_m = by_sex.mean_wing[by_sex.Sex .== "M"][1]
by_sex

In [ ]:
#| label: grouped-numbers
#| echo: false
#| output: false
wing_gap = wing_m - wing_f

# How far apart are the two piles, really? Two different questions.
n_f_above = count(>(wing_m), females.Wing)      # females longer than the male average
n_m_below = count(<(wing_f), males.Wing)        # males shorter than the female average

band_lo = max(minimum(females.Wing), minimum(males.Wing))
band_hi = min(maximum(females.Wing), maximum(males.Wing))
n_band = count(w -> band_lo <= w <= band_hi, sparrows.Wing)

**Itchy:** Read that as three steps stuck together. `groupby` cuts the frame into piles by a column. `combine` walks the piles and computes something on each. And the arrow pairs say what to compute and what to call it: take `Wing`, apply `mean`, name the answer `mean_wing`. `nrow => :n` counts the pile.

Then the two lines under it, which pull one number out of that table and carry the most important piece of Julia in this book. `by_sex.Sex .== "F"` has a **dot in front of the operator**, and a dot means *do this to every element*: without it `==` would ask whether the whole column is the letter F; with it, every entry is asked in turn and you get back trues and falses, one per row. Square brackets around that vector keep the rows where it is true, and the `[1]` takes the first — here the only — one out, so `wing_f` is a number and not a vector of length one. The dot works on any operator and, from next week, on any function. Once you can read it you can read most of the code in this book.

**Toto:** Males have longer wings.

**Itchy:** Males average `{julia} round(wing_m, digits = 2)` millimetres, females `{julia} round(wing_f, digits = 2)`, a gap of `{julia} round(wing_gap, digits = 2)`. And now the sentence I want you to be uncomfortable with: **that gap is not a finding yet.** It is two averages. It has no interval on it — no range of values it could plausibly be — it does not know that males are also bigger overall, and it cannot tell you whether the difference is in the wing or in the whole bird. Every one of those repairs is a chapter. Today you have two numbers, honestly computed, and that is genuinely all you have. Draw it.

In [ ]:
#| label: fig-sexes
#| fig-cap: "The same cloud, split by sex: the shaded band is the wing-length range both sexes share, the dashed lines are the two group means."
fig = Figure(size = (540, 380))
ax = Axis(fig[1, 1]; xlabel = "tarsus (mm)", ylabel = "wing (mm)",
    title = "the same cloud, with sex marked")
# the shared band goes down FIRST, low-opacity, so every point still sits on
# top of it — this is band_lo:band_hi from the cell above, drawn rather than
# only stated, so "n_band of 171 birds fall in a range both sexes occupy" is
# something the reader sees, not just a number in the prose
hspan!(ax, band_lo, band_hi; color = (LIGHT.muted, 0.15), label = "wing range shared by both sexes")
scatter!(ax, females.Tarsus, females.Wing; markersize = 7, label = "female")
scatter!(ax, males.Tarsus, males.Wing; markersize = 7, marker = :diamond, label = "male")
hlines!(ax, [wing_f]; linestyle = :dash, color = Cycled(1))
hlines!(ax, [wing_m]; linestyle = :dash, color = Cycled(2))
# legend outside the plot, so it cannot sit on top of the points it labels
fig[1, 2] = Legend(fig, ax; framevisible = false)
fig

**Momo:** The two clouds are separated, but they are also sitting in each other's laps.

**Itchy:** Which is exactly the right way to be confused, so let me make the confusion into two numbers instead of one. At the level of **averages** the two piles are quite well separated: only `{julia} n_f_above` of the `{julia} n_females` females have a wing longer than the average male, and only `{julia} n_m_below` of the `{julia} n_males` males have one shorter than the average female. But at the level of an **individual bird**, `{julia} n_band` of the `{julia} n_birds` sparrows have a wing length that occurs in both sexes.

**Toto:** So you could not sex a bird from its wing.

**Itchy:** Not most birds, no, and that is the whole distinction. A difference between group averages can be clean, real and worth reporting, and still leave most single animals inside a range both groups occupy. Those are two different claims and people run them together constantly. Hold the picture; it is what sits underneath half the arguments in the literature about effect sizes — how big a difference is, as opposed to whether one exists.

### Reading the red

**Itchy:** Right. Everybody close whatever else is open. This section is the reason this class exists on the first morning rather than the fifth, and it is the single most cited gap in every book that tries to teach this language: nobody teaches you to read the errors. People do not abandon Julia because the statistics are hard. They abandon it because they hit thirty lines of red text with a word like `MethodError` in it, feel stupid, and go back to R. Nothing in those thirty lines is trying to make you feel stupid. Most of it is Julia telling you, in detail, what it *does* know how to do.

**Momo:** In R an error was one sentence.

**Itchy:** One sentence which told you nothing. Julia gives you a paragraph which tells you everything, and the skill is knowing which line of the paragraph matters. Four failures, in the order you will meet them. Toto, misspell a column.

In [ ]:
#| label: err-column
#| error: true
sparrows.Wingg

**Itchy:** One line of message, then — in your own REPL — a list headed `Stacktrace`. This book trims that list from its printed output, so below you will see the placeholder `(stack trace omitted)`; ignore it either way. Read the one line to its end. `ArgumentError` — you handed a function an argument it cannot use. `column name "Wingg" not found in the data frame` — that is the fact. And then the part people skip because they have already started retyping: **`existing most similar names are: "Wing"`**. Julia went through your column names, worked out how close each one was to what you typed, and handed back the near miss. The fix is inside the error.

**Toto:** So the last clause is the answer.

**Itchy:** The last clause is the answer, and remember that it is only there because somebody could compute it. Watch what happens when nobody can. Momo, load a package.

In [ ]:
#| label: err-package
#| error: true
using DataFrame

**Momo:** But we are already using data frames.

**Itchy:** You are, and the package is called `DataFrames`, with an `s`. Now read the two lines, and read them differently from each other. Line one is a **fact**: there is no package by that name where Julia looks. Line two, `Run import Pkg; Pkg.add("DataFrame")`, is a **guess** — it is the sentence Julia prints for every missing name, and it does not know whether you failed to install something or misspelled something you already have. Here it is a misspelling, so following the advice would send you hunting a package that does not exist while the one you want sits already loaded.

**Eddie:** That is the opposite of the column error.

**Itchy:** It is exactly the opposite, and that is why I put them next to each other. In the first, Julia checked the near misses and told you. In the second, it did not check and offered a default. **The last line of an error is not automatically the fix.** Sometimes it is a diagnosis, sometimes it is a suggestion, and telling them apart is the skill. In your own REPL, look at the stack trace under this one too, because it is doing something useful by being useless: every frame in it is inside Julia's own package-loading machinery, and not one of them is a line you wrote — this page trims it to `(stack trace omitted)`, but that is what it hides. When a stack trace contains none of your code, the mistake is in *what you asked for*, not in what you did with it. Toto, add a number to a piece of text.

In [ ]:
#| label: err-type
#| error: true
sparrows.Tarsus[1] + "0.5"

**Toto:** That is the wall of red.

**Itchy:** That is the wall of red, and it has three parts. **Part one, the first line**: `no method matching +(::Float64, ::String)`. That is *what you asked for*, stated in types — you asked to add a number to a piece of text. **Part two, under `Closest candidates are:`** — that is *what exists*, the versions of `+` that came nearest to matching. **Part three, inside each candidate**, the argument that failed to match is printed **in red**. At a plain terminal with no colour, the same thing appears as the word `!Matched` written in front of that argument; it is the same marker either way.

**Momo:** Then the red should tell me which of my two arguments was wrong.

**Itchy:** It should, and on this particular error it does not, and I want you to see that on the first morning rather than the fifth. Look at what the candidates actually are: a four-argument version of `+`, and two from a package about automatic differentiation you have never heard of and did not ask for. The red on the first one sits on arguments you never even supplied. **This is the candidate list being useless**, and it is useless for a specific, predictable reason: `+` has hundreds of methods, so "the closest three" are close by a similarity rule and not by anything to do with your problem.

**Toto:** Then what do I read?

**Itchy:** The first line, and then you stop, because the first line is already a complete diagnosis: `::String`. Your second argument is text. And it is text because you wrote `"0.5"` with quotation marks round it, which is a piece of writing that looks like a number and is not one. That is not a contrived error, either — it is exactly what happens the first time you read a spreadsheet with a stray comma in a numeric column, and the type row in `first(sparrows, 5)` is where you would have caught it an hour earlier. **The rule is: when the function is enormous and general, read the first line. When it is small and specific, the candidate list is worth its weight.** Which brings us to the one I warned you about when you first met `filter`.

In [ ]:
#| label: err-order
#| error: true
filter(sparrows, row -> row.Sex == "M")

**Momo:** That is the reversal you warned about.

**Itchy:** It is, and now you have seen the punishment. Same three parts, and this time the candidate list earns its keep, because `filter` is a small specific function rather than `+`. First line: `no method matching filter(::DataFrame, ::var"#...")`, and that unreadable `var"#..."` is simply Julia's internal name for the anonymous function you typed — you never named it, so Julia made a name up. Then the candidates. Now do what you just learned: find the red.

**Toto:** It is on the second argument in all three of them. And the first argument is `::Any` every time.

**Itchy:** Which together is a complete diagnosis, and you can read it without knowing one thing about `filter`. **Every** version of this function will take anything at all in slot one, and **every** version insists on a collection in slot two. So the thing that failed is your second argument, and your second argument was a function. Your collection and your function are in the wrong slots. Turn them round, which is the call you already wrote correctly ten minutes ago, and it works.

**Momo:** I would have stared at that for an hour.

**Itchy:** Everybody does, once. Here is the routine, three questions, always in this order. **What did I ask for?** — the first line, in types. **What exists?** — the candidate list, when the function is specific enough for the list to mean anything. **Which argument is wrong?** — the one printed in red, or marked `!Matched` where there is no colour. Almost every Julia error you meet this year answers at least the first of those, and most answer all three, and reading them in that order means you never have to understand the rest of the wall.

**Toto:** Every one of ours just says `(stack trace omitted)`. What was under it?

**Itchy:** A list of who called whom, innermost call first — this page trims it, but in your own REPL it is there. The line that points at **you** is the one that says `top-level scope`, which in a notebook names the cell you ran. Everything above it is the machinery you called on the way down. For a mistake made at the prompt, that one line is all of the stack trace that is yours.

**Eddie:** And when the mistake is not yours?

**Itchy:** Then `top-level scope` is at the bottom as usual, but the interesting frames are the ones above it, inside somebody's package — and you have found either a bug or an assumption you did not know you were breaking. That is a good day. Not today.

### A number that is the same tomorrow

**Itchy:** Last piece. Some of what this book does is random — a jitter in a plot, a residual in Class 3 (a residual is what is left of each bird after the model has had its say), a whole appendix of simulations — and random is a problem for a book, because a book has to say the same thing twice. So we make randomness that repeats. `MersenneTwister` is a random-number generator, named after the arithmetic inside it; the number in the brackets is the seed, the starting point it counts from.

In [ ]:
#| label: seed-first
rng_sim = MersenneTwister(20260907)

# A coin, flipped n times. The generator is a keyword with no default, so it
# is impossible to call this function and forget to say which stream of
# randomness you meant. That is the house rule, in one line of code.
flip(n; rng) = rand(rng, Bool, n)

n_flips = 1000
run_a = flip(n_flips; rng = rng_sim)
p_a = mean(run_a)

@printf("flips: %d   proportion heads: %.4f\n", n_flips, p_a)

**Toto:** I got `{julia} round(p_a, digits = 3)`, which is not a half.

**Itchy:** Before the number, the line above it, which is the first function you have written. `flip(n; rng) = rand(rng, Bool, n)` **defines a function**: a name, its arguments, an equals sign, the expression it returns — all a function needs when its body is one line. After the semicolon comes a keyword, as in a call, and this one has no default, so `flip` cannot be called without naming a generator. The comment states the house rule; the language enforces it. And `rand(rng, Bool, n)` reads "from `rng`, draw `n` trues or falses", which is a coin.

Now the number. It is not a half, and it should not be. `{julia} n_flips` flips of a fair coin lands near a half, not on it, and how near is a question with an actual answer that Appendix A works out. What matters this morning is the other thing: run it again.

In [ ]:
#| label: seed-again
run_b = flip(n_flips; rng = MersenneTwister(20260907))
run_c = flip(n_flips; rng = MersenneTwister(20260908))

p_b = mean(run_b)
p_c = mean(run_c)

@printf("same seed     : proportion %.4f   identical draws: %s\n", p_b, run_a == run_b)
@printf("seed + one    : proportion %.4f   identical draws: %s\n", p_c, run_a == run_c)

**Momo:** The same seed gave the same number exactly.

**Itchy:** Not the same *number*. The same **draws** — all `{julia} n_flips` of them, in the same order, which is why the proportion could not help but agree. And one digit later, at the seed one greater, you get `{julia} round(p_c, digits = 3)` instead of `{julia} round(p_a, digits = 3)`, from a completely different set of flips. So be precise about what a seed buys you: **a seed promises reproducibility, not correctness.** It makes your accident the same accident every time, so that somebody else can have your accident too. It does not make the accident representative. If your conclusion changes when you change the seed, the seed was doing your arguing for you.

**Eddie:** Which is why you would run it at many seeds.

**Itchy:** Which is exactly why, and that is Appendix A's whole subject. Draw the two runs.

In [ ]:
#| label: fig-coin
#| fig-cap: "Running proportion of heads for two seeds. Both wander, both settle, neither lands on a half."
running(v) = cumsum(v) ./ (1:length(v))

fig = Figure(size = (540, 380))
ax = Axis(fig[1, 1]; xlabel = "flips so far", ylabel = "proportion heads",
    title = "two seeds, one coin")
hlines!(ax, [0.5]; linestyle = :dot, color = (:grey, 0.8))
lines!(ax, 1:n_flips, running(run_a); linewidth = 2, label = "seed 20260907")
lines!(ax, 1:n_flips, running(run_c); linewidth = 2, linestyle = :dash, label = "seed 20260908")
# full 0-1 range: the first few flips really do sit at 0 or 1, and clipping them
# to a narrower window would hide exactly the wildness this figure is about.
ax.limits = (nothing, nothing, 0.0, 1.0)
fig[1, 2] = Legend(fig, ax; framevisible = false)
fig

**Itchy:** Both lines are wild on the left, where a handful of flips can say anything, and both calm down towards the dotted line without ever quite arriving. That picture is the reason a small sample can tell you almost anything you want to hear, and it is the same picture underneath every confidence interval in this book — the range of values a sample leaves you unable to rule out. Appendix A turns it into arithmetic. In the cell, `1:n_flips` is a **range**, the whole numbers from one to a thousand without writing them out, and `./` is the dot again, dividing each running total by its own count.

### The formula you are not going to fit

**Itchy:** One last thing, and then you are free. Next week's model has a shape, and I want you to have seen the shape before you meet the model. In Julia a model formula is a **value**, like a number or a vector, that you can make and look at on its own. `@formula` is the morning's second macro, and the `@` is why it can take `Wing ~ Tarsus` as you would write it on paper: a function would try to evaluate `Wing` and find no such thing.

In [ ]:
#| label: formula-object
f = @formula(Wing ~ Tarsus)

**Toto:** It printed the formula back at me and called both columns unknown.

**Itchy:** Read that word `unknown` slowly, because it is the whole lesson. Julia made an object that says "wing, as a function of tarsus", and it is sitting there in `f` knowing nothing about sparrows. It has not seen the data — it cannot have, you never handed it any — so it does not yet know whether `Wing` is a number, a category, or a column that does not exist. `unknown` is not a failure. It is the formula honestly reporting that it has not been introduced to anybody yet. Now put it in the box this book uses.

In [ ]:
#| label: formula-box
bf(f)

**Momo:** There is a `sigma` in there and I did not type one.

**Itchy:** You did not, and that is the most useful thing on this page. `bf` is the "big formula" box, and it always has two slots. The `mu` slot holds what you wrote: the mean of wing depends on tarsus. The `sigma` slot holds `1`, which means "the spread is one number, the same for every bird" — a constant, and you never chose it, and today you cannot even see why you would want to. **Class 4 is the week you put something in that second slot**, and the box has been quietly holding it open since your first hour. The `nothing` at the end is a third slot, for a second response column — used when the response is a count of successes out of so many trials, which a later class reaches — and it stays empty until then.

**Toto:** So what do I do with it?

**Itchy:** Nothing, today. You hand it to `drm` with a family — the shape of distribution you are willing to assume for the response, Normal today, others from Class 3 on — and some data, and it becomes a fitted model, and that is Class 2's first ten minutes. I am stopping here on purpose: you can now get a file in, look at it four ways, ask it a question, draw it, make a random number that behaves, and read the message when it breaks. That is base camp. Everything above it is one climb.

---

## Summary

### Stats stuff

- **Look before you model, four ways, and know what each look is for.** The size tells you the file is the file you meant. The first rows tell you the **types**, which is where a stray comma in a numeric column shows up as `String`. The summary tells you the ranges and the missing counts. The picture tells you the shape, which the summary cannot: identical means and standard deviations are consistent with wildly different distributions.
- **Draw the cloud before you fit anything to it.** The habit costs four seconds and prevents fitting a straight line to something that is visibly not straight. Class 2 draws the same cloud again before it puts a line through it, and every chapter after that keeps the order.
- **A difference between group means is not yet a finding, and a clean difference between means is not a clean difference between animals.** Ours was a gap of `{julia} round(wing_gap, digits = 1)` millimetres between male and female wings, and it is clean at the level of averages: only `{julia} n_f_above` of the `{julia} n_females` females beat the average male. It is not clean at the level of birds: most of the sample sits inside a band of wing lengths that both sexes occupy, so you could not sex a bird from its wing. Report the first without implying the second. And everything that turns the gap into a claim — an interval, controlling for overall body size, admitting that repeated measurements on one bird are not independent — is a later chapter.
- **When you split a dataset, add the pieces back up.** One line, and it catches the filter that silently dropped the rows where a value was missing.
- **A seed promises reproducibility, not correctness.** The same seed gives the same draws in the same order, so anyone can reproduce your accident exactly. A different seed gives a different accident with a similar answer. If a conclusion moves when the seed moves, the seed is doing the arguing, and the repair is many seeds, which is Appendix A.
- **Reading an error is a statistical skill, not a computing chore.** The evidence is not folklore: developers spend a substantial fraction of a debugging task simply *reading* error messages, and reading them is about as hard as reading source code (Barik et al. 2017). Time spent learning the shape of these messages is time bought back on every chapter after this one.

### Julia you used

Each of these appeared for the first time in this class, and the book leans on every one from here on.

- **The dot: `.==`, `./`, and any operator or function.** A dot in front of an operator means *do this to every element*, and gives back a vector. `by_sex.Sex .== "F"` asks every row whether it is a female and returns trues and falses; `cumsum(v) ./ (1:length(v))` divides element by element. This is the single most-used piece of Julia in the book.
- **The colon: `:Wing`, `:mean`.** A **Symbol** is a name written as a value. It lets you hand a function the *name* of a column or a statistic without Julia trying to look it up. DataFrames takes Symbols wherever it refers to a column.
- **The `@`: `@printf`, `@formula`.** A **macro**, which rewrites the code it is given before running it. `@printf` fills a pattern — `%d` a whole number, `%.4f` four decimals, `\n` a line end — from the values after the comma; `@formula` accepts `Wing ~ Tarsus` written as on paper.
- **The semicolon in a call: `Axis(fig[1, 1]; xlabel = ...)`.** What follows it is a **keyword argument**: named, any order, optional unless the function says otherwise.
- **The `!`: `hist!`, `scatter!`.** A convention meaning "this changes something you gave it" — here, it draws onto an existing axis.
- **`row -> row.Sex == "M"`.** An **anonymous function**: "given a row, is its Sex M". `==("F")` is the same idea for one operator with one side left open, and `ByRow` applies it to every row.
- **`v[mask]` and `[1]`.** Square brackets with a vector of trues and falses keep the rows where it is true; a trailing `[1]` takes the first element out, so you get a number rather than a vector of length one.
- **`lo, hi = extrema(x)`.** A function can return several values, and a comma-separated left-hand side catches them all.
- **`flip(n; rng) = rand(rng, Bool, n)`.** A **function definition** in one line: name, arguments, `=`, the expression it returns. A keyword after the `;` with no default must be supplied by the caller, which is how this book makes it impossible to forget the generator.
- **`1:n_flips`.** A **range**: the whole numbers from one to *n*, without writing them out.
- **Errors, in three questions.** *What did I ask for?* — the first line, in types. *What exists?* — the list under `Closest candidates are:`, worth reading only when the function is narrow (`filter`, not `+`). *Which argument is wrong?* — the one printed **in red**, or marked `!Matched` where there is no colour. In the `Stacktrace`, `top-level scope` is the line that is yours. The last line is not automatically the fix: a near-miss computed from your columns is a diagnosis; `Pkg.add("X")` is a default printed for every missing name.

### Calls you used

- `using Pkg1, Pkg2`: makes packages visible and compiles what it needs the first time in a session; the pause happens once. `include("../tools/figures.jl")` runs the book's own helper file.
- `CSV.read("path.csv", DataFrame)`: `CSV` parses, `DataFrames` holds. Paths are relative to the file being run.
- `nrow(df)`, `ncol(df)`, `names(df)`, `first(df, 5)`, `describe(df, :mean, :std, :min, :max, :nmissing)`: the four looks. Read the type row under `first`; R prints no equivalent.
- `df.Wing`: a column, as a plain `Vector`, which any function that takes a vector will take.
- `filter(row -> ..., df)`: **function first, data second**, the reverse of `dplyr::filter`. `subset(df, :Sex => ByRow(==("F")))`: data first.
- `combine(groupby(df, :Sex), :Wing => mean => :mean_wing, nrow => :n)`: group, then compute per group; the double arrow reads "take this column, apply this function, call the answer this".
- `fig_data_cloud(x, y; xlabel, ylabel, title)` from `tools/figures.jl`: the house data cloud, the first figure of the grammar. `Figure`, `Axis`, `scatter!`, `lines!`, `hist!`, `hlines!`, `vlines!`, `Legend`: the CairoMakie verbs the house figures are built from.
- `@formula(y ~ x)` and `bf(...)`: the formula as a value, and the "big formula" box `drm` takes, with its `mu` slot holding what you wrote and its `sigma` slot holding the constant `1` until Class 4.
- `MersenneTwister(seed)` and `rand(rng, Bool, n)`: a named stream of random numbers, and *n* coin flips from it. **Every draw in this book takes its generator as an `rng` argument** — `rand(rng_sim, Bool, n)` today, `residuals(fit; type = :quantile, rng = rng_sim)` and `simulate(fit; nsim = 200, rng = rng_sim)` from Class 2 on, the same keyword in all three — never `Random.seed!`, which sets a hidden switch for the whole session.

---

## Further reading

*Graded by depth. References checked 2026-09-07.*

1. **Bezanson, J., Edelman, A., Karpinski, S. & Shah, V. B. (2017) "Julia: A Fresh Approach to Numerical Computing", *SIAM Review* 59:65–98.** doi:10.1137/141000671. The language, explained by the people who built it. Read it for one idea — that a function is a collection of methods chosen by the types of *all* its arguments — because that idea is exactly what the `Closest candidates` list in this chapter's errors is printing.
2. **Bouchet-Valat, M. & Kamiński, B. (2023) "DataFrames.jl: Flexible and Fast Tabular Data in Julia", *Journal of Statistical Software* 107(4).** doi:10.18637/jss.v107.i04. The reference for everything in this chapter's middle third, by the maintainers. Read the sections on `groupby`/`combine` when the double-arrow pairs stop looking like syntax and start looking like a grammar.
3. **Barik, T., Smith, J., Lubick, K., Holmes, E., Feng, J., Murphy-Hill, E. & Parnin, C. (2017) "Do Developers Read Compiler Error Messages?", *2017 IEEE/ACM 39th International Conference on Software Engineering (ICSE)*, 575–585.** doi:10.1109/ICSE.2017.59. An eye-tracking study of 56 people fixing real defects. Its findings are the reason this class spends an hour of its first morning on red text: participants did read the messages, reading them was about as hard as reading source code, that difficulty predicted whether they solved the task, and it took 13–25% of their total time.
4. **Becker, B. A., Denny, P., Pettit, R., Bouchard, D., Bouvier, D., Harrington, B., Kamil, A., Karkare, A., McDonald, C., Osera, P.-M., Pearce, J. L. & Prather, J. (2019) "Compiler Error Messages Considered Unhelpful: The Landscape of Text-Based Programming Error Message Research", *ITiCSE-WGR '19*, 177–210.** doi:10.1145/3344429.3372508. Half a century of research on error messages, gathered in one place. Read it if you ever have to teach this to somebody else: it is a survey of what has been tried, including what does not work.
5. **Danisch, S. & Krumbiegel, J. (2021) "Makie.jl: Flexible high-performance data visualization for Julia", *Journal of Open Source Software* 6(65):3349.** doi:10.21105/joss.03349. The plotting library under every figure in this book. Worth skimming once so that `Figure`, `Axis` and the trailing exclamation marks stop being incantations.
6. **The Julia manual, "Getting Started" and "Noteworthy Differences from R"** (<https://docs.julialang.org/>). Not a paper, and the single most useful thing on this list for its first week. The differences page is a list of exactly the traps this chapter had room to show you only one of.

---

## Exercises

Graded by depth: the first three take ten minutes each. Every exercise names a file under `data/` that exists; do it on your own organism as well where you have one. A *check* is a number computed from that file when this page was built — match it before going on. **For every question, paste your code and then explain in your own words what each line does**, as if to somebody who has not opened Julia. The worked answer under question 3 shows what that explanation should look like.

In [ ]:
#| label: exercise-checks
#| echo: false
#| output: false
# Numbers the exercise checks quote, computed from the named files.
ex_bs = CSV.read("data/2012/BodySize.csv", DataFrame)
ex_bs_rows, ex_bs_cols = nrow(ex_bs), ncol(ex_bs)
ex_bs_text = [n for n in names(ex_bs) if eltype(ex_bs[!, n]) <: AbstractString]
ex_an = CSV.read("data/2012/Anscombe.csv", DataFrame)
ex_an_tab = combine(groupby(ex_an, :Type), :Food => mean => :mean_food,
                    :Drink => mean => :mean_drink, :Drink => std => :sd_drink, nrow => :n)
ex_an_mean_lo, ex_an_mean_hi = extrema(ex_an_tab.mean_drink)
ex_an_sd_lo, ex_an_sd_hi = extrema(ex_an_tab.sd_drink)
ex_ch = CSV.read("data/2012/ChickSurvival.csv", DataFrame)
ex_ch_tab = combine(groupby(ex_ch, :Sex), :Mass2 => mean => :mean_mass, nrow => :n)
ex_mass_f = ex_ch_tab.mean_mass[ex_ch_tab.Sex .== "F"][1]
ex_mass_m = ex_ch_tab.mean_mass[ex_ch_tab.Sex .== "M"][1]
ex_p1 = mean(flip(n_flips; rng = MersenneTwister(1)))
ex_p2 = mean(flip(n_flips; rng = MersenneTwister(2)))

1. **Get a file in, and read the type row.** Read `data/2012/BodySize.csv` with `CSV.read`. Report `nrow`, `ncol` and `names`, then run `first(df, 5)` and write one sentence about the **type row**: is every column the type you expected? Some measurement columns are not, and the reason is in the file. *Check:* `{julia} ex_bs_rows` rows, `{julia} ex_bs_cols` columns, and `{julia} length(ex_bs_text) - 2` of the six measurement columns printing as text rather than `Float64`. Class 6 opens with what to do about it.

2. **Four looks, and the one the summary cannot give.** `data/2012/Anscombe.csv` has three columns: `Food`, `Drink` and a `Type` with four levels. Run `describe`, then `combine(groupby(df, :Type), :Drink => mean => :mean_drink, :Drink => std => :sd_drink, nrow => :n)`. Then draw four data clouds with `fig_data_cloud`, one per type, using `filter` to pick each. Name what the pictures show that the table cannot. *Check:* all four means round to `{julia} round(ex_an_mean_hi, digits = 2)` and all four SDs to `{julia} round(ex_an_sd_hi, digits = 2)`; the four clouds are not the same shape, and only one of them is a cloud.

3. **Split and add up.** On the chapter's file, split the sparrows into those with a tarsus above the mean and those at or below it, using `filter` for one half and `subset` for the other. Report both counts and check that they sum to `nrow(sparrows)`. *Check:* `{julia} count(>(mean(sparrows.Tarsus)), sparrows.Tarsus)` above and `{julia} count(<=(mean(sparrows.Tarsus)), sparrows.Tarsus)` at or below, summing to `{julia} n_birds`.

**A worked answer, and its marking.** Toto handed this in. His code first, then his explanation, then what Itchy wrote in the margin.

In [ ]:
#| label: worked-answer
big = filter(row -> row.Tarsus > mean(sparrows.Tarsus), sparrows)
small = subset(sparrows, :Tarsus => ByRow(<=(mean(sparrows.Tarsus))))
@printf("above: %d   at or below: %d   total: %d\n", nrow(big), nrow(small), nrow(big) + nrow(small))

**Toto:** *Line one keeps every row whose tarsus is bigger than the average tarsus; the arrow makes a small function that asks the question of one row, and `filter` asks it of every row. Line two does the same for the other half with the other verb: `ByRow` asks the question row by row, and `<=(...)` is "is it at most this". Line three prints the two counts and their sum, so I can see they add up to `{julia} n_birds`.*

**Itchy, in the margin:** Three lines, three sentences, and each sentence says what the line *does* rather than repeating what it *says* — that is the standard. One correction. In line one you wrote "the average tarsus" as if it were a number you knew; it is `mean(sparrows.Tarsus)`, computed on the spot, and if you had misspelled the column there the error would have come from `mean`, not from `filter`. Say where each number comes from. And one thing you did not say: the two conditions are `>` and `<=`, which between them cover every bird exactly once. That is *why* the sum has to come out right, and the sentence I most wanted was that one.

4. **A group mean, and its limits.** `data/2012/ChickSurvival.csv` holds `{julia} nrow(ex_ch)` chicks with a `Sex` column and their mass at two days old, `Mass2`. Compute the mean mass per sex with `combine(groupby(df, :Sex), :Mass2 => mean => :mean_mass, nrow => :n)`. Then draw the chicks as a cloud of `Mass2` against `JulianDate` (the hatching date) with the sexes marked, as the chapter's second figure did. Two sentences: what the numbers say, and what the picture says that the numbers do not. *Check:* the two means are `{julia} round(ex_mass_f, digits = 3)` g and `{julia} round(ex_mass_m, digits = 3)` g, and the gap between them is smaller than the gap between any two chicks you could point at.

5. **Break it five ways.** On the chapter's file, reproduce each of the four errors: a misspelled column, a package name that does not exist, a number added to a piece of text, and `filter` with its arguments the wrong way round. For each, quote the **one line** of the message that told you what to fix, and say whether that line was a diagnosis or a guess. Then make a fifth mistake this chapter did not cover — any mistake — and answer the three questions on it: what did you ask for, what exists, which argument was wrong. *Check:* the first two are `ArgumentError`s and the last two `MethodError`s; the chapter's verdict on each one's last line is on the page, so compare yours with it.

6. **Two seeds.** Adapt the coin-flip cell to `MersenneTwister(1)` and `MersenneTwister(2)`. Report the proportion of heads for seed 1, for seed 1 run a second time, and for seed 2. Then answer in one sentence: which of those three numbers was guaranteed to equal which, and why "the same seed gives the same answer" is a weaker statement than what actually happened. *Check:* seed 1 gives `{julia} round(ex_p1, digits = 3)` both times, seed 2 gives `{julia} round(ex_p2, digits = 3)`.

7. **The box that is already holding a slot open.** Write `bf(@formula(Weight ~ Tarsus))` for the chapter's sparrows and print it. Say what is in the `mu` slot, what is in the `sigma` slot, and — in one sentence, guessing is fine — what kind of scientific question would make you want to put something other than a constant in the second one. *Check:* the printout has three slots; the second holds `1` and the third holds `nothing`, and you typed neither.